<a href="https://colab.research.google.com/github/kulanigayathree/J26-IT-364/blob/model%2FFishing-Zone-Prediction/notebooks/C2_fishing_zone_prediction/C2_2_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C2 – Notebook 2: Baselines (simple methods the deep models must beat)

| Method | How it guesses | Uses |
|---|---|---|
| **Climatology** | "In this month, boats usually fish here" | fishing history 2017–21 |
| **Persistence** | "Boats fished here last week, so again" | **recent** fishing data — shown only as an *upper reference*, because the real system has no live fishing data |
| **Gradient Boosting** | classic ML, one cell at a time | the **same ocean inputs** as the deep models (fair comparison) |

Same data, same time split (train 2017–21, validate 2022, test 2023–24) and same scores as the deep-learning notebooks.
**Main score = PR-AUC** (higher is better; fishing cells are rare).

No GPU needed. `Runtime → Run all`.

## Step 1 – Connect Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
BASE = '/content/drive/MyDrive/C2_data'
CKPT_ROOT = f'{BASE}/checkpoints'; os.makedirs(CKPT_ROOT, exist_ok=True)
OUT = f'{BASE}/results';           os.makedirs(OUT, exist_ok=True)
LOCAL_CUBE = '/content/datacube_0p25.nc'
if not os.path.exists(LOCAL_CUBE):                 # copy once to Colab's fast disk (much quicker than reading Drive)
    shutil.copy(f'{BASE}/datacube_0p25.nc', LOCAL_CUBE)
print('Datacube ready:', LOCAL_CUBE)

Mounted at /content/drive
Datacube ready: /content/datacube_0p25.nc


## Step 2 – Settings (same for every model)

In [2]:
import numpy as np, pandas as pd, xarray as xr, time, json, pickle
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '(turn on the T4 GPU if this says cpu)')
torch.manual_seed(42); np.random.seed(42)

# ---- same for EVERY model (fair comparison) ----
T_IN, LEADS = 7, 3                     # 7 input days (ending yesterday) -> Day 1, Day 2, Day 3
TRAIN = ('2017-01-01', '2021-12-31')
VAL   = ('2022-01-01', '2022-12-31')
TEST  = ('2023-01-01', '2024-12-31')
FEATURES = ['sst', 'sst_grad', 'log_chl', 'sla', 'u', 'v', 'cur_speed', 'mld',
            'depth', 'moon_sin', 'moon_cos', 'doy_sin', 'doy_cos',
            'fish_clim']               # usual fishing grounds (monthly map from 2017-21 only)
EPOCHS, BATCH, LR, DROPOUT = 30, 16, 1e-3, 0.2
PATIENCE = 5                           # early stopping
MC_RUNS, CAL_RUNS = 30, 10             # MC dropout runs for test / for fitting calibration

Device: cpu (turn on the T4 GPU if this says cpu)


## Step 3 – Load the datacube and build the inputs

In [3]:
X = xr.open_dataset(LOCAL_CUBE).load()
days = pd.DatetimeIndex(X.time.values)
ocean = X['depth'].isel(time=0).notnull().values                   # (H, W) True = sea
label = np.nan_to_num(X['fishing'].values).astype('float32')       # (time, H, W) 1 = fishing
tr = (days >= TRAIN[0]) & (days <= TRAIN[1])

# Usual fishing grounds: how often each cell was fished in this month, TRAINING years only.
# It is history, not real-time data, so the model never needs live fishing data.
clim_m = np.stack([label[tr & (days.month == m)].mean(0) for m in range(1, 13)])     # (12, H, W)
X['fish_clim'] = (('time', 'latitude', 'longitude'),
                  np.where(ocean, clim_m[np.asarray(days.month) - 1], np.nan).astype('float32'))

# Features -> (time, channel, H, W), normalised with TRAINING statistics only
feat = np.stack([X[v].values for v in FEATURES], axis=1).astype('float32')
mean = np.nanmean(feat[tr], axis=(0, 2, 3), keepdims=True)
std  = np.nanstd(feat[tr],  axis=(0, 2, 3), keepdims=True) + 1e-6
feat = np.nan_to_num((feat - mean) / std, nan=0.0)                 # land / gaps -> 0

NT, C, H, W = feat.shape
lat, lon = X.latitude.values, X.longitude.values
print(f'{NT} days, {C} input channels, grid {H}x{W}, sea cells {ocean.sum()}')
print('Fishing rate (training years):', round(float(label[tr][:, ocean].mean()), 4))
json.dump({'features': FEATURES, 'mean': mean.ravel().tolist(), 'std': std.ravel().tolist()},
          open(f'{OUT}/normalisation.json', 'w'))

2922 days, 14 input channels, grid 40x44, sea cells 1421
Fishing rate (training years): 0.0046


## Step 4 – Samples and the scoring function

In [4]:
first = np.arange(T_IN - 1, NT - LEADS)                  # t = last input day ("yesterday")
tgt_day = days[first + 1]                                # Day 1 = t + 1 ("today")
def idx(period): return first[(tgt_day >= period[0]) & (tgt_day <= period[1])]
idx_tr, idx_va, idx_te = idx(TRAIN), idx(VAL), idx(TEST)
print('samples  train', len(idx_tr), ' val', len(idx_va), ' test', len(idx_te))

FEAT_T, LAB_T = torch.from_numpy(feat), torch.from_numpy(label)
class Windows(torch.utils.data.Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        t = self.ids[i]
        return FEAT_T[t - T_IN + 1: t + 1], LAB_T[t + 1: t + 1 + LEADS], t

dl_tr = torch.utils.data.DataLoader(Windows(idx_tr), batch_size=BATCH, shuffle=True, drop_last=True)
dl_va = torch.utils.data.DataLoader(Windows(idx_va), batch_size=BATCH)
dl_te = torch.utils.data.DataLoader(Windows(idx_te), batch_size=BATCH)

def scores(prob, ids):
    # prob: (n, LEADS, H, W). Scores sea cells only. PR-AUC is the main score (fishing cells are rare).
    rows = []
    for k in range(LEADS):
        y = label[ids + 1 + k][:, ocean].ravel()
        p = np.clip(prob[:, k][:, ocean].ravel(), 1e-6, 1 - 1e-6)
        rows.append({'lead_day': k + 1, 'PR_AUC': average_precision_score(y, p),
                     'ROC_AUC': roc_auc_score(y, p), 'Brier': brier_score_loss(y, p)})
    return pd.DataFrame(rows).round(4)

samples  train 1819  val 365  test 729


## Step 5 – Climatology and Persistence

In [5]:
def clim_pred(ids):
    return np.stack([np.stack([clim_m[days[t + 1 + k].month - 1] for k in range(LEADS)]) for t in ids])
def pers_pred(ids):              # mean fishing of the 7 input days (needs live fishing data!)
    p = np.stack([label[t - T_IN + 1: t + 1].mean(0) for t in ids])
    return np.repeat(p[:, None], LEADS, axis=1)

val_rows, test_rows = [], []
for name, fn in [('Climatology', clim_pred), ('Persistence (upper ref.)', pers_pred)]:
    v = scores(fn(idx_va), idx_va); v.insert(0, 'model', name); val_rows.append(v)
    s = scores(fn(idx_te), idx_te); s.insert(0, 'model', name); test_rows.append(s)
    print(f'\n{name} (test 2023-24)'); print(s.drop(columns='model').to_string(index=False))


Climatology (test 2023-24)
 lead_day  PR_AUC  ROC_AUC  Brier
        1  0.2307   0.6272 0.1349
        2  0.2307   0.6272 0.1349
        3  0.2308   0.6271 0.1350

Persistence (upper ref.) (test 2023-24)
 lead_day  PR_AUC  ROC_AUC  Brier
        1  0.5914   0.8649 0.0785
        2  0.5659   0.8506 0.0823
        3  0.5526   0.8440 0.0843


## Step 6 – Gradient Boosting (classic ML, same ocean inputs)
Each sea cell on day *t* is one row of 14 numbers; the model predicts fishing on *t + 1*. It cannot see
neighbouring cells or earlier days — that is the difference the deep models are tested against.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
rng = np.random.default_rng(0)
yy_all, xx_all = np.where(ocean)
def pixel_table(ids, n=None):
    t = np.repeat(ids, len(yy_all)); yy = np.tile(yy_all, len(ids)); xx = np.tile(xx_all, len(ids))
    if n and n < len(t):
        s = rng.choice(len(t), n, replace=False); t, yy, xx = t[s], yy[s], xx[s]
    return feat[t, :, yy, xx], t, yy, xx

Xp, t_, y_, x_ = pixel_table(idx_tr, n=400_000)            # 400k random training cells (speed)
hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0)
hgb.fit(Xp, label[t_ + 1, y_, x_])

def hgb_pred(ids):
    Xq, tq, yq, xq = pixel_table(ids)
    out = np.zeros((len(ids), LEADS, H, W), 'float32')
    p = hgb.predict_proba(Xq)[:, 1]
    pos_ = np.searchsorted(ids, tq)
    for k in range(LEADS): out[pos_, k, yq, xq] = p        # same per-cell forecast for each lead day
    return out

v = scores(hgb_pred(idx_va), idx_va); v.insert(0, 'model', 'Gradient Boosting'); val_rows.append(v)
s = scores(hgb_pred(idx_te), idx_te); s.insert(0, 'model', 'Gradient Boosting'); test_rows.append(s)
print('Gradient Boosting (test 2023-24)'); print(s.drop(columns='model').to_string(index=False))

Gradient Boosting (test 2023-24)
 lead_day  PR_AUC  ROC_AUC  Brier
        1  0.2286   0.6320 0.1356
        2  0.2286   0.6319 0.1356
        3  0.2283   0.6317 0.1357


## Step 7 – Save

In [7]:
pd.concat(val_rows).to_csv(f'{OUT}/baselines_val_scores.csv', index=False)
pd.concat(test_rows).to_csv(f'{OUT}/baselines_test_scores.csv', index=False)
print(pd.concat(test_rows).to_string(index=False))
print('\nSaved results/baselines_val_scores.csv and results/baselines_test_scores.csv')

                   model  lead_day  PR_AUC  ROC_AUC  Brier
             Climatology         1  0.2307   0.6272 0.1349
             Climatology         2  0.2307   0.6272 0.1349
             Climatology         3  0.2308   0.6271 0.1350
Persistence (upper ref.)         1  0.5914   0.8649 0.0785
Persistence (upper ref.)         2  0.5659   0.8506 0.0823
Persistence (upper ref.)         3  0.5526   0.8440 0.0843
       Gradient Boosting         1  0.2286   0.6320 0.1356
       Gradient Boosting         2  0.2286   0.6319 0.1356
       Gradient Boosting         3  0.2283   0.6317 0.1357

Saved results/baselines_val_scores.csv and results/baselines_test_scores.csv
